# ConvNeXt-Small + MIL com Optuna (Ordinal Focal Loss)

Treina ConvNeXt-Small com:
- **MIL (Gated Attention pooling)** sobre bags de patches histopatológicos
- **BCEFocalOrdinalLoss** combinando focal loss + penalidade ordinal
- **Optuna** para busca dos pesos ótimos da perda (`gamma`, `w_focal`, `w_ord`)

In [1]:
import sys
sys.path.append('../../../')

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler, SequentialSampler
from torchvision.models import convnext_small, ConvNeXt_Small_Weights
import albumentations as Albu
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score, precision_score
from tqdm import tqdm
import optuna
from optuna.pruners import MedianPruner
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from utils.dataset import PandasWithMilDataset
from utils.mil import ConvNeXtMIL

## Configuração

In [2]:
# Reprodutibilidade
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Hiperparâmetros fixos
BATCH_SIZE      = 4
NUM_WORKERS     = 4
OUTPUT_CLASSES  = 5
INIT_LR         = 1e-4
WARMUP_FACTOR   = 2
WARMUP_EPOCHS   = 1
N_EPOCHS        = 50
DROPOUT_RATE    = 0.4
PATIENCE        = 8
MAX_PATCHES     = 36

# Optuna
N_OPTUNA_TRIALS = 30
N_OPTUNA_EPOCHS = 8

# Caminhos
ROOT_DIR    = '../../..'
IMAGES_DIR  = '/home/woshington/Projects/Doutorado/bag_of_patches'

os.makedirs('logs',   exist_ok=True)
os.makedirs('models', exist_ok=True)

MODEL_PATH = 'models/convnext-small-mil-optuna.pth'
LOG_PATH   = 'logs/convnext-small-mil-optuna.txt'

print(f'Images dir: {IMAGES_DIR}')

Device: cuda
Images dir: /home/woshington/Projects/Doutorado/bag_of_patches


## Função de Perda

In [3]:
class BCEFocalOrdinalLoss(nn.Module):
    """
    Perda combinada: Focal Loss + Penalidade Ordinal.

    loss = w_focal * focal + w_ord * ordinal_mse

    Args:
        gamma:   expoente focal (controla foco em exemplos difíceis).
        w_focal: peso do termo focal.
        w_ord:   peso do termo ordinal (MSE sobre classe esperada).
    """

    def __init__(self, gamma: float = 2.0, w_focal: float = 1.0, w_ord: float = 1.0):
        super().__init__()
        self.gamma   = gamma
        self.w_focal = w_focal
        self.w_ord   = w_ord

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.to(logits.device).float()
        probs   = torch.sigmoid(logits)

        # Focal loss
        bce         = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t         = probs * targets + (1 - probs) * (1 - targets)
        focal_loss  = ((1 - p_t) ** self.gamma * bce).mean()

        # Penalidade ordinal (MSE sobre classe esperada)
        expected_cls = probs.sum(dim=1)
        target_cls   = targets.sum(dim=1)
        max_cls      = logits.shape[1]
        ord_loss     = ((expected_cls - target_cls) ** 2).mean() / (max_cls ** 2)

        return self.w_focal * focal_loss + self.w_ord * ord_loss


print('BCEFocalOrdinalLoss definida.')

BCEFocalOrdinalLoss definida.


## Carregamento de Dados

In [4]:
def remove_nonexistent(df, images_dir):
    mask = df['image_id'].apply(lambda x: os.path.isdir(os.path.join(images_dir, x)))
    return df[mask].reset_index(drop=True)


df_all = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_all.columns = df_all.columns.str.strip()

# Filtragem por entropia (remove 20% mais difíceis)
df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_entropy = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove   = int(len(df_entropy) * 0.2)
ids_remove = set(df_entropy.head(n_remove)['image_id'])
df_all     = df_all[~df_all['image_id'].isin(ids_remove)].reset_index(drop=True)

train_idx = np.where(df_all['fold'] != 3)[0]
valid_idx = np.where(df_all['fold'] == 3)[0]

df_train = df_all.loc[train_idx].reset_index(drop=True)
df_val   = df_all.loc[valid_idx].reset_index(drop=True)
df_test  = pd.read_csv(f'{ROOT_DIR}/data/test.csv')
df_test.columns = df_test.columns.str.strip()

df_train = remove_nonexistent(df_train, IMAGES_DIR)
df_val   = remove_nonexistent(df_val,   IMAGES_DIR)
df_test  = remove_nonexistent(df_test,  IMAGES_DIR)

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')
print('Distribuição de classes (train):')
print(df_train['isup_grade'].value_counts().sort_index())

Train: 7073 | Val: 1767 | Test: 1590
Distribuição de classes (train):
isup_grade
0    1953
1    1802
2     899
3     826
4     832
5     761
Name: count, dtype: int64


## Augmentação

In [5]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.Resize(224, 224),
])

val_transforms = Albu.Compose([
    Albu.Resize(224, 224),
])

## Funções de Treino e Validação

In [ ]:
def training_step(model, dataloader, optimizer, device, loss_fn, scaler):
    model.train()
    losses = []
    for bag, mask, targets, _ in dataloader:
        bag     = bag.to(device, non_blocking=True)
        mask    = mask.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out  = model(bag, mask)
            loss = loss_fn(out['logits'], targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.detach().cpu().item())
    return losses


def validation_step(model, dataloader, device, loss_fn):
    model.eval()
    val_loss, all_preds, all_targets = [], [], []
    with torch.no_grad():
        for bag, mask, targets, _ in dataloader:
            bag     = bag.to(device, non_blocking=True)
            mask    = mask.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                out  = model(bag, mask)
                loss = loss_fn(out['logits'], targets)
            probs       = torch.sigmoid(out['logits'])
            preds       = (probs > 0.5).sum(dim=1)
            targets_cls = targets.sum(dim=1).long()
            all_preds.append(preds.cpu())
            all_targets.append(targets_cls.cpu())
            val_loss.append(loss.cpu().item())
    all_preds   = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    return {
        'val_loss':      np.mean(val_loss),
        'val_acc':       accuracy_score(all_targets, all_preds),
        'val_kappa':     cohen_kappa_score(all_targets, all_preds, weights='quadratic'),
        'val_f1':        f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_recall':    recall_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_precision': precision_score(all_targets, all_preds, average='macro', zero_division=0),
    }

## Busca de Hiperparâmetros com Optuna

O Optuna busca os melhores valores de `gamma`, `w_focal` e `w_ord` para a `BCEFocalOrdinalLoss`
treinando por `N_OPTUNA_EPOCHS` épocas e maximizando o **Kappa quadrático** na validação.

In [ ]:
# DataLoaders para a busca Optuna (reutilizados em cada trial)
optuna_train_ds = PandasWithMilDataset(
    IMAGES_DIR, df_train, transforms=train_transforms,
    normalize=True, max_patches=MAX_PATCHES
)
optuna_val_ds = PandasWithMilDataset(
    IMAGES_DIR, df_val, transforms=val_transforms,
    normalize=True, max_patches=MAX_PATCHES
)

optuna_train_loader = DataLoader(
    optuna_train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    sampler=RandomSampler(optuna_train_ds),
    pin_memory=True, drop_last=True,
)
optuna_val_loader = DataLoader(
    optuna_val_ds, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    sampler=SequentialSampler(optuna_val_ds),
    pin_memory=True,
)

print(f'Optuna train batches: {len(optuna_train_loader)}')
print(f'Optuna val   batches: {len(optuna_val_loader)}')

In [8]:
def build_model():
    backbone = convnext_small(weights=ConvNeXt_Small_Weights.DEFAULT)
    model = ConvNeXtMIL(
        model=backbone,
        output_classes=OUTPUT_CLASSES,
        unfreeze_last_blocks=2,
        dropout_rate=DROPOUT_RATE,
        hidden_dim=512,
        gated=True,
        pool='att',
    )
    return model.to(device)


def objective(trial: optuna.Trial) -> float:
    gamma   = trial.suggest_float('gamma',   0.5, 3.0)
    w_focal = trial.suggest_float('w_focal', 0.1, 2.0)
    w_ord   = trial.suggest_float('w_ord',   0.1, 2.0)

    model   = build_model()
    loss_fn = BCEFocalOrdinalLoss(gamma=gamma, w_focal=w_focal, w_ord=w_ord)
    opt     = optim.Adam(model.parameters(), lr=INIT_LR)
    scaler  = torch.amp.GradScaler()

    best_kappa = 0.0

    for epoch in range(N_OPTUNA_EPOCHS):
        training_step(model, optuna_train_loader, opt, device, loss_fn, scaler)
        metrics = validation_step(model, optuna_val_loader, device, loss_fn)
        kappa   = metrics['val_kappa']

        trial.report(kappa, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

        if kappa > best_kappa:
            best_kappa = kappa

    del model
    torch.cuda.empty_cache()
    return best_kappa


print('Objective definido. Iniciando estudo Optuna...')

Objective definido. Iniciando estudo Optuna...


In [ ]:
STUDY_DB   = 'sqlite:///logs/convnext-small-mil-optuna.db'
STUDY_NAME = 'convnext-small-mil-ordinal-focal'

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    storage=STUDY_DB,
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f'\nMelhor trial: #{best.number}')
print(f'  Kappa: {best.value:.4f}')
print(f'  gamma:   {best.params["gamma"]:.4f}')
print(f'  w_focal: {best.params["w_focal"]:.4f}')
print(f'  w_ord:   {best.params["w_ord"]:.4f}')

In [ ]:
# Importância dos hiperparâmetros
importances = optuna.importance.get_param_importances(study)
print('Importância dos hiperparâmetros:')
for param, imp in importances.items():
    print(f'  {param}: {imp:.4f}')

# Extrai melhores parâmetros
BEST_GAMMA   = best.params['gamma']
BEST_W_FOCAL = best.params['w_focal']
BEST_W_ORD   = best.params['w_ord']

## Dashboard e Visualizações Optuna

O estudo é persistido em SQLite — use as células abaixo para análise inline (Plotly)
ou inicie o dashboard web com o comando na última célula desta seção.

| Gráfico | O que mostra |
|---|---|
| `plot_optimization_history` | Evolução do Kappa ao longo dos trials |
| `plot_intermediate_values` | Valores intermediários (por época) com pruning |
| `plot_parallel_coordinate` | Correlação entre hiperparâmetros e objetivo |
| `plot_param_importances` | Importância relativa de cada hiperparâmetro |
| `plot_contour` | Superfície 2-D de pares de hiperparâmetros |
| `plot_slice` | Efeito marginal de cada hiperparâmetro |


In [ ]:
from optuna import visualization as optvis

fig = optvis.plot_optimization_history(study)
fig.update_layout(title='Histórico de Otimização — Kappa por Trial', height=450)
fig.show()


In [ ]:
fig = optvis.plot_intermediate_values(study)
fig.update_layout(title='Valores Intermediários por Época (trials com pruning)', height=450)
fig.show()


In [ ]:
fig = optvis.plot_parallel_coordinate(study, params=['gamma', 'w_focal', 'w_ord'])
fig.update_layout(title='Coordenadas Paralelas — Hiperparâmetros vs Kappa', height=500)
fig.show()


In [ ]:
fig = optvis.plot_param_importances(study)
fig.update_layout(title='Importância dos Hiperparâmetros (fANOVA)', height=400)
fig.show()


In [ ]:
# Pares de hiperparâmetros com maior impacto
for p1, p2 in [('gamma', 'w_focal'), ('gamma', 'w_ord'), ('w_focal', 'w_ord')]:
    fig = optvis.plot_contour(study, params=[p1, p2])
    fig.update_layout(title=f'Contorno: {p1} × {p2}', height=500)
    fig.show()


In [ ]:
fig = optvis.plot_slice(study, params=['gamma', 'w_focal', 'w_ord'])
fig.update_layout(title='Efeito Marginal de Cada Hiperparâmetro', height=450)
fig.show()


### Dashboard Web

Inicia o servidor Optuna Dashboard na porta 8080.

In [ ]:
# Lança o Optuna Dashboard no navegador (porta 8080 por padrão)
# Instale antes: pip install optuna-dashboard
# Execute na célula ou no terminal:

print(f'Storage: {STUDY_DB}')
print(f'Study  : {STUDY_NAME}')
print()
print('Para abrir o dashboard, execute no terminal:')
print(f'  optuna-dashboard {STUDY_DB}')
print()
print('Ou diretamente desta célula (abre em background):')


In [ ]:
import subprocess, threading

def _run_dashboard():
    subprocess.run(['optuna-dashboard', STUDY_DB], check=False)

thread = threading.Thread(target=_run_dashboard, daemon=True)
thread.start()
print('Dashboard iniciado em http://127.0.0.1:8080')
print('(o processo fica em background; reinicie o kernel para parar)')


## Treino Completo com Melhores Hiperparâmetros

In [10]:
train_dataset = PandasWithMilDataset(
    IMAGES_DIR, df_train, transforms=train_transforms,
    normalize=True, max_patches=MAX_PATCHES
)
valid_dataset = PandasWithMilDataset(
    IMAGES_DIR, df_val, transforms=val_transforms,
    normalize=True, max_patches=MAX_PATCHES
)
test_dataset  = PandasWithMilDataset(
    IMAGES_DIR, df_test, transforms=val_transforms,
    normalize=True, max_patches=MAX_PATCHES
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    sampler=RandomSampler(train_dataset),
    pin_memory=True, prefetch_factor=2, persistent_workers=True, drop_last=True,
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    sampler=SequentialSampler(valid_dataset),
    pin_memory=True, prefetch_factor=2, persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    shuffle=False, pin_memory=True, prefetch_factor=2, persistent_workers=True,
)

print(f'Train: {len(train_loader)} batches | Val: {len(valid_loader)} batches | Test: {len(test_loader)} batches')

Train: 1768 batches | Val: 221 batches | Test: 199 batches


In [ ]:
model = build_model()

loss_function = BCEFocalOrdinalLoss(
    gamma=BEST_GAMMA,
    w_focal=BEST_W_FOCAL,
    w_ord=BEST_W_ORD,
)

optimizer = optim.Adam(model.parameters(), lr=INIT_LR / WARMUP_FACTOR)

scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, N_EPOCHS - WARMUP_EPOCHS
)
scheduler = GradualWarmupScheduler(
    optimizer,
    multiplier=WARMUP_FACTOR,
    total_epoch=WARMUP_EPOCHS,
    after_scheduler=scheduler_cosine,
)

scaler = torch.amp.GradScaler()

print(f'Modelo: ConvNeXt-Small + MIL')
print(f'Perda: gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parâmetros treináveis: {trainable:,} / {total:,}')

In [ ]:
best_kappa = 0.0
best_epoch = 0
no_improve = 0

history = {
    'train_loss':    [],
    'val_loss':      [],
    'val_acc':       [],
    'val_kappa':     [],
    'val_f1':        [],
    'val_recall':    [],
    'val_precision': [],
}

print('Iniciando treino completo...')
print('=' * 80)

for epoch in range(1, N_EPOCHS + 1):
    print(f'\nÉpoca {epoch}/{N_EPOCHS}')

    train_losses = training_step(model, train_loader, optimizer, device, loss_function, scaler)
    metrics      = validation_step(model, valid_loader, device, loss_function)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(np.mean(train_losses))
    for k in ['val_loss', 'val_acc', 'val_kappa', 'val_f1', 'val_recall', 'val_precision']:
        history[k].append(metrics[k])

    print(f'  Train Loss: {history["train_loss"][-1]:.5f}')
    print(f'  Val   Loss: {metrics["val_loss"]:.5f} | Acc: {metrics["val_acc"]*100:.2f}% | Kappa: {metrics["val_kappa"]:.4f} | F1: {metrics["val_f1"]:.4f}')
    print(f'  LR: {current_lr:.2e}')

    log_line = (f'epoch: {epoch} | lr: {current_lr:.2e} | '
                f'train_loss: {history["train_loss"][-1]:.5f} | '
                f'val_loss: {metrics["val_loss"]:.5f} | '
                f'val_acc: {metrics["val_acc"]:.4f} | '
                f'val_kappa: {metrics["val_kappa"]:.4f}\n')
    with open(LOG_PATH, 'a') as f:
        f.write(log_line)

    if metrics['val_kappa'] > best_kappa:
        best_kappa = metrics['val_kappa']
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  ✓ Melhor modelo salvo! Kappa: {best_kappa:.4f}')
    else:
        no_improve += 1
        print(f'  Sem melhora por {no_improve} época(s)')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping na época {epoch}. Melhor: época {best_epoch} (Kappa={best_kappa:.4f})')
        break

print('\nTreino concluído!')
print(f'Melhor Kappa de validação: {best_kappa:.4f} na época {best_epoch}')

## Curvas de Aprendizado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'],   label='Val Loss')
axes[0, 0].set_title('Loss')
axes[0, 0].set_xlabel('Época')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history['val_acc'], color='green', label='Val Accuracy')
axes[0, 1].set_title('Acurácia de Validação')
axes[0, 1].set_xlabel('Época')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history['val_kappa'], color='orange', label='Val Kappa')
axes[1, 0].set_title('Kappa Quadrático de Validação')
axes[1, 0].set_xlabel('Época')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history['val_f1'], color='red', label='Val F1')
axes[1, 1].set_title('F1 Macro de Validação')
axes[1, 1].set_xlabel('Época')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('logs/convnext-small-mil-optuna-training.png', dpi=300, bbox_inches='tight')
plt.show()

## Avaliação no Conjunto de Teste

In [ ]:
# from code.tests.others.vit-base-mil-optuna import build_model
model = build_model()
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for bag, mask, targets, _ in tqdm(test_loader, desc='Testing'):
        bag  = bag.to(device,  non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = model(bag, mask)
        probs       = torch.sigmoid(out['logits'])
        preds       = (probs > 0.5).sum(dim=1)
        targets_cls = targets.sum(dim=1).long()
        all_preds.append(preds.cpu())
        all_targets.append(targets_cls.cpu())

all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

test_acc   = accuracy_score(all_targets, all_preds)
test_kappa = cohen_kappa_score(all_targets, all_preds, weights='quadratic')
test_f1    = f1_score(all_targets, all_preds, average='macro', zero_division=0)

print('=' * 60)
print('RESULTADOS NO TESTE')
print('=' * 60)
print(f'Acurácia : {test_acc*100:.2f}%')
print(f'Kappa    : {test_kappa:.4f}')
print(f'F1 Macro : {test_f1:.4f}')
print('=' * 60)
print(f'Hiperparâmetros Optuna usados:')
# print(f'  gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}')

Testing: 100%|██████████| 199/199 [02:07<00:00,  1.56it/s]

RESULTADOS NO TESTE
Acurácia : 66.16%
Kappa    : 0.8454
F1 Macro : 0.6122
Hiperparâmetros Optuna usados:


NameError: name 'BEST_GAMMA' is not defined

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

classes = ['ISUP 0', 'ISUP 1', 'ISUP 2', 'ISUP 3', 'ISUP 4', 'ISUP 5']

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_norm, annot=True, fmt='.3f', cmap='Blues',
    xticklabels=classes, yticklabels=classes,
)
plt.title('Matriz de Confusão Normalizada — ConvNeXt-Small MIL')
plt.ylabel('Classe Verdadeira')
plt.xlabel('Classe Prevista')
plt.tight_layout()
plt.savefig('logs/convnext-small-mil-optuna-confusion-matrix.png', dpi=300, bbox_inches='tight')
plt.show()